In [1]:
pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.1 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

from catboost import CatBoostRegressor

In [3]:
# LOAD DATA
df = pd.read_csv("/content/used_device_data.csv")

print("Dataset Shape:", df.shape)

Dataset Shape: (3454, 15)


In [4]:
# FEATURE ENGINEERING
CURRENT_YEAR = 2026

df["device_age"] = CURRENT_YEAR - df["release_year"]

df["usage_ratio"] = (
    df["days_used"] /
    (df["device_age"] * 365 + 1)
)

df["camera_total"] = (
    df["rear_camera_mp"] +
    df["front_camera_mp"]
)

df["battery_per_weight"] = (
    df["battery"] /
    df["weight"]
)

df["ram_storage_ratio"] = (
    df["ram"] /
    df["internal_memory"]
)

In [5]:
# Features & TARGET
X = df.drop(
    columns=["normalized_used_price"]
)
y = df["normalized_used_price"]

In [6]:
# CATEGORICAL FEATURES
cat_features = [
    "device_brand",
    "os",
    "4g",
    "5g"
]

In [7]:
# TRAIN TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [8]:
# CATBOOST MODel
model = CatBoostRegressor(
    iterations=2000,
    learning_rate=0.03,
    depth=8,
    loss_function="RMSE",
    eval_metric="R2",
    random_seed=42,
    verbose=100
)

In [9]:
# TRAIN
model.fit(
    X_train,
    y_train,
    cat_features=cat_features,
    eval_set=(X_test, y_test),
    use_best_model=True
)

0:	learn: 0.0425847	test: 0.0428429	best: 0.0428429 (0)	total: 61.4ms	remaining: 2m 2s
100:	learn: 0.8605211	test: 0.8422071	best: 0.8422071 (100)	total: 1.11s	remaining: 20.9s
200:	learn: 0.8872129	test: 0.8558139	best: 0.8558139 (200)	total: 2.14s	remaining: 19.1s
300:	learn: 0.9001209	test: 0.8575166	best: 0.8575460 (299)	total: 3.15s	remaining: 17.8s
400:	learn: 0.9111475	test: 0.8577283	best: 0.8578718 (382)	total: 4.67s	remaining: 18.6s
500:	learn: 0.9197459	test: 0.8571140	best: 0.8578718 (382)	total: 6.47s	remaining: 19.4s
600:	learn: 0.9278444	test: 0.8566577	best: 0.8578718 (382)	total: 7.6s	remaining: 17.7s
700:	learn: 0.9346334	test: 0.8560058	best: 0.8578718 (382)	total: 10.4s	remaining: 19.3s
800:	learn: 0.9403206	test: 0.8552495	best: 0.8578718 (382)	total: 12.4s	remaining: 18.6s
900:	learn: 0.9453809	test: 0.8547106	best: 0.8578718 (382)	total: 13.7s	remaining: 16.7s
1000:	learn: 0.9501769	test: 0.8544824	best: 0.8578718 (382)	total: 15.6s	remaining: 15.6s
1100:	learn: 

CatBoostRegressor(depth=8, eval_metric='R2', iterations=2000, learning_rate=0.03, loss_function='RMSE', random_seed=42, verbose=100)

In [10]:
# PREDICT
y_pred = model.predict(X_test)

In [11]:
# EVALUATE
r2 = r2_score(y_test, y_pred)

mae = mean_absolute_error(
    y_test,
    y_pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)

print("\n===== CATBOOST RESULTS =====")

print("R2 :", round(r2, 4))
print("MAE:", round(mae, 4))
print("RMSE:", round(rmse, 4))


===== CATBOOST RESULTS =====
R2 : 0.8579
MAE: 0.1716
RMSE: 0.2148


In [12]:
# FEATURE IMPORTANCE
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.get_feature_importance()
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print("\n===== TOP FEATURES =====")
print(importance.head(20))


===== TOP FEATURES =====
                 Feature  Importance
13  normalized_new_price   39.998927
2            screen_size   11.148981
7        internal_memory    7.274714
10                weight    5.409142
16          camera_total    5.251602
9                battery    4.930166
6        front_camera_mp    4.147730
5         rear_camera_mp    4.124226
18     ram_storage_ratio    3.167607
1                     os    2.438956
17    battery_per_weight    1.891869
0           device_brand    1.704945
3                     4g    1.478389
11          release_year    1.450850
15           usage_ratio    1.409420
8                    ram    1.392949
14            device_age    1.375853
12             days_used    1.109176
4                     5g    0.294495


In [13]:
# SAMPLE PREDICTION
sample = X_test.iloc[[0]]

pred = model.predict(sample)[0]

print("\nPredicted Log Price :", round(pred, 4))
print("Actual Log Price    :", round(y_test.iloc[0], 4))

print(
    "\nPredicted Price : $",
    round(np.exp(pred), 2)
)

print(
    "Actual Price    : $",
    round(np.exp(y_test.iloc[0]), 2)
)


Predicted Log Price : 4.0537
Actual Log Price    : 3.9742

Predicted Price : $ 57.61
Actual Price    : $ 53.21
